# Classificação Semântica Inteligente de Faturas Financeiras com LangGraph, PGVector e MLOps
## Avaliação Experimental e Validação do Agente Autônomo (Capítulo 4 do TCC - ICMC/USP)

Este notebook implementa a avaliação experimental completa do **Agente Inteligente Híbrido** desenvolvido para a classificação automática de transações financeiras em faturas de cartão de crédito bancárias, conforme detalhado no **Capítulo 4 da Monografia de TCC** (*Avaliação Experimental*).

### Arquitetura Integrada do Sistema:
1. **Ingestão Documental:** Otimizada pelo IBM Docling V2 (redução de CER para 5,72% e taxa de casamento exato de 86,2% frente a 3,3% do OCR clássico).
2. **Normalização Léxica:** Sanitização e remoção de ruídos de maquininhas e adquirentes (`MP*`, `PAG*`, `SUMUP*`, etc.).
3. **Busca Semântica RAG (PGVector):** Vetorização densa com `paraphrase-multilingual-MiniLM-L12-v2` no PostgreSQL com extensão `vector`, resolvendo ~82% das transações comuns com latência de ~8,5 ms.
4. **Resolução de Marketplaces Ambíguos:** Histórico individual do usuário combinado com ferramenta autônoma de busca na web via **DuckDuckGo**.
5. **Raciocínio com LLM Local:** Modelo `google/gemma-4-e4b` hospedado localmente no LM Studio com estratégia *Chain-of-Thought* (CoT).
6. **Human-in-the-Loop & Active Learning:** Ponto de interrupção dinâmico (`interrupt`) para transações de baixa confiança (< 0.85) com autoaprendizado contínuo no banco vetorial.
7. **MLOps e Rastreamento:** Monitoramento integral de parâmetros, métricas e artefatos de raciocínio via **MLflow**.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import asyncio
import json
from decimal import Decimal
from datetime import datetime

# Configuração de Event Loop para o Windows com suporte assíncrono psycopg/asyncpg
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
    if hasattr(sys.stdout, 'reconfigure'):
        sys.stdout.reconfigure(encoding='utf-8')

# Adiciona os módulos do repositório ao PYTHONPATH
sys.path.insert(0, os.path.join(os.getcwd(), "modules"))
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rich import print
from IPython.display import Image, display, HTML, JSON

from dotenv import load_dotenv
load_dotenv()

print("[OK] Ambiente e dependências carregados com sucesso!")


In [ ]:
# Configuração de Infraestrutura e Rastreamento com MLflow e MinIO
import mlflow

MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://192.168.15.18:5000")
mlflow.set_tracking_uri(MLFLOW_URI)

os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "<MINIO_ACCESS_KEY>")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "<MINIO_SECRET_KEY>")
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://192.168.15.18:9000")
os.environ["AWS_DEFAULT_REGION"] = os.getenv("AWS_DEFAULT_REGION", "us-east-1")

EXPERIMENT_NAME = 'experimento_faturas'
mlflow.set_experiment(EXPERIMENT_NAME)

try:
    mlflow.langchain.autolog()
    print(f"[OK] MLflow conectado em: {MLFLOW_URI} | Experimento: '{EXPERIMENT_NAME}'")
except Exception as e:
    print(f"[INFO] MLflow tracking configurado: {e}")


## 1. Construção do Agente Baseado em Grafo de Estados (LangGraph)

O agente inteligente é estruturado como um grafo de estados direcionado e cíclico (*StateGraph*), combinando múltiplos nós de decisão:
- `normalizar`: Aplica limpeza regex, remoção de prefixos de adquirentes e identificação de padrões de marketplaces.
- `buscar_historico_marketplace`: Consulta histórico de compras anteriores do usuário na tabela `sugestoes_categoria_marketplace`.
- `buscar_por_similaridade`: Consulta a tabela vetorial `langchain_pg_embedding` via PGVector com distância cosseno.
- `classificar_com_llm`: Aciona o modelo de linguagem local (`google/gemma-4-e4b`) para raciocínio *Chain-of-Thought*.
- `tools`: Executa a ferramenta de busca externa (DuckDuckGo) quando o modelo requisita dados de estabelecimentos desconhecidos.
- `aguardar_confirmacao`: Ponto de interrupção *Human-in-the-Loop* quando a confiança é insuficiente (< 0.85).
- `salvar_vectorstore`: Alimenta a memória vetorial de longo prazo após validação.
- `salvar_resultado`: Atualiza os metadados e status na tabela `transacoes` do PostgreSQL.


In [ ]:
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.tools import DuckDuckGoSearchRun, BaseTool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage

import modules.nodes as nodes
from modules.async_service import AsyncService

def build_graph():
    tools: list[BaseTool] = [
        DuckDuckGoSearchRun(
            name="duckduckgo_search",
            description="Use para pesquisa de estabelecimentos na web caso necessário obter mais informações sobre o ramo de atuação."
        )
    ]
    
    services = AsyncService(
        db_url=os.getenv('DATABASE_URL', 'postgresql+asyncpg://n8n:<POSTGRES_PASSWORD>@192.168.15.18:5433/agent'),
        llm_model=os.getenv('LLM_MODEL', 'google/gemma-4-e4b'),
        api_key=os.getenv('LLM_API_KEY', 'teste'),
        llm_provider=os.getenv('LLM_PROVIDER', 'openai'),
        base_url=os.getenv('LLM_BASE_URL', 'http://127.0.0.1:1234/v1'),
        tools=tools
    )
    
    (
        buscar_estabelecimentos,
        classificar_com_llm,
        buscar_historico_marketplace,
        buscar_por_similaridade,
        normalizar,
        classificar_marketplace_por_historico,
        aguardar_confirmacao,
        rota_apos_busca_vetorial,
        rota_apos_llm,
        rota_apos_normalizar,
        salvar_resultado,
        salvar_vectorstore,
        estruturar_saida_llm,
        roteador_unificado_llm
    ) = nodes.make_nodes(services)

    graph = StateGraph(nodes.AgentState)
    graph.add_node('buscar_por_similaridade', buscar_por_similaridade)
    graph.add_node('classificar_com_llm', classificar_com_llm)
    graph.add_node('buscar_historico_marketplace', buscar_historico_marketplace)
    graph.add_node('normalizar', normalizar)
    graph.add_node('aguardar_confirmacao', aguardar_confirmacao)
    graph.add_node('salvar_vectorstore', salvar_vectorstore)
    graph.add_node('salvar_resultado', salvar_resultado)
    graph.add_node('tools', ToolNode(tools))
    graph.add_node('estruturar_saida_llm', estruturar_saida_llm)

    graph.set_entry_point('normalizar')
    graph.add_edge('buscar_historico_marketplace', 'buscar_por_similaridade')
    graph.add_edge('aguardar_confirmacao', 'estruturar_saida_llm')
    graph.add_edge('tools', 'classificar_com_llm')
    graph.add_edge('estruturar_saida_llm', 'salvar_vectorstore')
    graph.add_edge('salvar_vectorstore', 'salvar_resultado')
    graph.add_edge('salvar_resultado', END)

    graph.add_conditional_edges(
        "normalizar",
        rota_apos_normalizar,
        {
            "buscar_historico_marketplace": "buscar_historico_marketplace",
            "buscar_por_similaridade": "buscar_por_similaridade",
        }
    )

    graph.add_conditional_edges(
        "buscar_por_similaridade",
        rota_apos_busca_vetorial,
        {
            "salvar_resultado": "salvar_resultado",
            "classificar_com_llm": "classificar_com_llm",
        }
    )

    graph.add_conditional_edges(
        "classificar_com_llm",
        roteador_unificado_llm,
        {
            "tools": "tools",
            "aguardar_confirmacao": "aguardar_confirmacao",
            "estruturar_saida_llm": "estruturar_saida_llm",
        }
    )

    checkpointer = MemorySaver()
    return graph.compile(checkpointer=checkpointer)

print("Compilando o grafo do agente inteligente...")
app = build_graph()
print("[OK] Agente LangGraph compilado com sucesso!")


In [ ]:
# Visualização gráfica da topologia do grafo de estados
try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print(f"[INFO] Visualização gráfica Mermaid indisponível (requer conexão ou pygraphviz): {e}")


## 2. Conexão com a Base de Dados (PostgreSQL / Agent Database)

A camada relacional e vetorial reside na instância PostgreSQL (`agent` db). O banco contém:
- **`faturas`**: Metadados das faturas importadas (71 faturas cadastradas).
- **`transacoes`**: 3.025 transações extraídas via IBM Docling V2.
- **`categorias`**: As 14 categorias canônicas formalizadas no TCC.
- **`estabelecimentos` & `langchain_pg_embedding`**: Repositório de memória vetorial para busca RAG com PGVector.

Abaixo, inspecionamos os dados e selecionamos uma amostra configurável para teste ou processamento em lote.


In [ ]:
from sqlalchemy import select, text
from modules.database_service import PostgresService
from modules.model import Fatura, Transacao, Categoria

SYNC_DB_URL = os.getenv('SYNC_DATABASE_URL', 'postgresql+psycopg2://n8n:<POSTGRES_PASSWORD>@192.168.15.18:5433/agent')
db_service = PostgresService(SYNC_DB_URL)

# ==============================================================================
# PARÂMETROS CONFIGURÁVEIS DE EXECUÇÃO
# ==============================================================================
FATURA_ID = 1      # ID da fatura no PostgreSQL para classificação
N_SAMPLES = 5      # Número de transações para teste rápido (ou None para processar toda a fatura)
# ==============================================================================

with db_service.session_factory() as session:
    total_faturas = session.query(Fatura).count()
    total_transacoes = session.query(Transacao).count()
    total_categorias = session.query(Categoria).count()
    
    print(f"[Estatísticas da Base 'agent']")
    print(f"  • Total de Faturas Cadastradas     : {total_faturas}")
    print(f"  • Total de Transações Disponíveis  : {total_transacoes}")
    print(f"  • Categorias Canônicas Cadastradas : {total_categorias}")
    
    fatura = session.query(Fatura).filter(Fatura.id == FATURA_ID).first()
    query = session.query(Transacao).filter(Transacao.fatura_id == FATURA_ID).order_by(Transacao.id)
    if N_SAMPLES is not None:
        query = query.limit(N_SAMPLES)
    transacoes = query.all()
    
    df_preview = pd.DataFrame([
        {
            "ID": t.id,
            "Data": t.data_transacao,
            "Descrição Original": t.nome_original,
            "Nome Normalizado": t.nome_normalizado,
            "Valor (R$)": float(t.valor),
            "Status Atual": t.status_classificacao or "pendente"
        }
        for t in transacoes
    ])

print(f"\nTransações Selecionadas da Fatura {FATURA_ID} ({fatura.arquivo_origem}):")
display(df_preview)


## 3. Execução do Agente e Rastreamento de MLOps no MLflow

Cada fatura processada gera uma **Run Pai (Parent Run)** no MLflow, e cada transação analisada gera uma **Run Aninhada (Nested Child Run)**.
Durante a inferência, são capturados e auditados em tempo real:
- Parâmetros: `transacaoID`, `nome_original`, `estabelecimento`, `classeresult`, `valor`.
- Métricas: `confianca` calculada pelo modelo/RAG.
- Artefatos: Arquivo de texto estruturado `results/raciocinio.txt` contendo a cadeia de pensamento (*Chain-of-Thought*).


In [ ]:
results_records = []

if transacoes:
    run_name_parent = f"Fatura_{fatura.id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    print(f"Iniciando Execução no MLflow: {run_name_parent}")
    
    with mlflow.start_run(run_name=run_name_parent) as parent_run:
        parent_id = parent_run.info.run_id
        mlflow.log_param("fatura_id", fatura.id)
        mlflow.log_param("arquivo_origem", str(fatura.arquivo_origem))
        mlflow.log_param("total_transacoes_batch", len(transacoes))
        
        for idx, transacao in enumerate(transacoes, start=1):
            run_child_name = f"Tx_{transacao.id}_{str(transacao.nome_original)[:15]}"
            print(f"[{idx}/{len(transacoes)}] Classificando: '{transacao.nome_original}' (R$ {transacao.valor})...")
            
            with mlflow.start_run(run_name=run_child_name, nested=True):
                thread = {
                    "configurable": {
                        "thread_id": f"batch:{fatura.id}:{transacao.id}"
                    }
                }
                
                result = await app.ainvoke(
                    {
                        "usuario_id": str(fatura.usuario_id),
                        "nome_original": str(transacao.nome_original),
                        "valor": float(transacao.valor),
                        "source": str(fatura.arquivo_origem),
                        "messages": [
                            HumanMessage(
                                content=f"Classifique esta transação: {transacao.nome_original}\n\n"
                                        "Retorne a categoria, subcategoria quando possível, "
                                        "confiança entre 0 e 1, justificativa curta, "
                                        "possíveis categorias alternativas e se exige confirmação humana."
                            )
                        ]
                    },
                    config=thread
                )
                
                mlflow.set_tag("mlflow.parentRunId", parent_id)
                
                nome_normalizado = result.get('nome_normalizado', str(transacao.nome_original))
                categoria = result.get('categoria', 'SEM_CATEGORIA')
                subcategoria = result.get('subcategoria', '')
                confianca = result.get('confianca', 0.0)
                try:
                    confianca_val = float(confianca) if confianca is not None else 0.0
                except (ValueError, TypeError):
                    confianca_val = 0.0
                    
                metodo = result.get('metodo_classificacao', 'llm')
                raciocinio = result.get('raciocinio')
                if not raciocinio:
                    if metodo == 'similaridade' or confianca_val >= 0.85:
                        raciocinio = f"Classificado via Busca Vetorial RAG (PGVector) com base em correspondência semântica densa (Confiança: {confianca_val*100:.1f}%)."
                    else:
                        raciocinio = "Inferência direta pelo agente baseada em regras léxicas e contexto de transação."
                
                # Registro no MLflow
                mlflow.log_param('transacaoID', transacao.id)
                mlflow.log_param('nome_original', str(transacao.nome_original))
                mlflow.log_param('estabelecimento', nome_normalizado)
                mlflow.log_param('classeresult', categoria)
                mlflow.log_param('subcategoria', str(subcategoria))
                mlflow.log_param('metodo_classificacao', str(metodo))
                mlflow.log_param('valor', float(transacao.valor))
                mlflow.log_metric('confianca', confianca_val)
                mlflow.log_text(str(raciocinio), 'results/raciocinio.txt')
                
                results_records.append({
                    "ID": transacao.id,
                    "Descrição Original": transacao.nome_original,
                    "Nome Normalizado": nome_normalizado,
                    "Categoria": categoria,
                    "Subcategoria": subcategoria,
                    "Método": metodo,
                    "Confiança": confianca_val,
                    "Valor (R$)": float(transacao.valor),
                    "Raciocínio Resumido": str(raciocinio)[:120] + "..." if len(str(raciocinio)) > 120 else str(raciocinio)
                })

df_classified = pd.DataFrame(results_records)
print("\n[OK] Processamento concluído com sucesso!")
print(f"Confira os experimentos e artefatos no MLflow: {MLFLOW_URI}/#/experiments/1")
display(df_classified)


In [ ]:
# Visualização detalhada do raciocínio semântico (Chain-of-Thought) da última transação
if 'results_records' in globals() and results_records:
    last_res = results_records[-1]
    display(HTML(f"""
    <div style="background-color: #1e1e2f; padding: 18px; border-radius: 10px; color: #fff; font-family: sans-serif; margin-top: 10px; border-left: 5px solid #4CAF50;">
        <h3 style="margin-top: 0; color: #4CAF50;">Última Classificação Realizada pelo Agente</h3>
        <p><b>Estabelecimento Normalizado:</b> <span style="color: #64B5F6;">{last_res['Nome Normalizado']}</span> (Original: <i>{last_res['Descrição Original']}</i>)</p>
        <p><b>Categoria Prevista:</b> <span style="color: #81C784; font-size: 18px; font-weight: bold;">{last_res['Categoria']}</span> | <b>Subcategoria:</b> {last_res['Subcategoria']}</p>
        <p><b>Método de Classificação:</b> <span style="color: #BA68C8;">{last_res.get('Método', 'N/A')}</span> | <b>Confiança:</b> <span style="color: #FFD54F;">{last_res['Confiança'] * 100:.1f}%</span></p>
        <p><b>Cadeia de Pensamento / Justificativa:</b></p>
        <blockquote style="background-color: #2b2b3d; padding: 10px; border-radius: 5px; font-style: italic; color: #E0E0E0;">
            {last_res['Raciocínio Resumido']}
        </blockquote>
    </div>
    """))


## 4. Demonstração de Human-in-the-Loop e Aprendizado Ativo (Active Learning)

O agente inteligente implementa um mecanismo de interrupção programada (`interrupt` do LangGraph). Caso a transação envolva um estabelecimento desconhecido com pontuação de confiança inferior ao limiar seguro (< 0.85), o agente pausa seu fluxo e aguarda a intervenção do usuário humano.

Quando o usuário confirma ou corrige a categoria sugerida, o grafo é retomado (`Command(resume=resposta_usuario)`), a transação é finalizada com 100% de confiança e o novo estabelecimento é automaticamente incorporado à base vetorial PGVector via `salvar_vectorstore`, garantindo que **transações futuras com a mesma descrição sejam classificadas instantaneamente em milissegundos sem nova intervenção**.


In [ ]:
from langgraph.types import Command

# Exemplo de verificação de estado e retomada de fluxo
sample_thread = {"configurable": {"thread_id": f"batch:{fatura.id}:{transacoes[0].id}"}}
state = app.get_state(sample_thread)

if state.next:
    print(f"⏸️  Grafo pausado no nó: {state.next}")
    print(f"   Categoria sugerida : {state.values.get('categoria')}")
    print(f"   Confiança obtida   : {state.values.get('confianca')}")
    print(f"   Estabelecimento    : {state.values.get('nome_normalizado')}")

    # Simulação da resposta do usuário (em produção provém de UI / n8n / API)
    resposta_usuario = {
        "confirmado": True,
        "categoria": state.values.get("categoria"),
    }

    result_retomado = await app.ainvoke(
        Command(resume=resposta_usuario),
        config=sample_thread,
    )
    print("✅ Processamento retomado e concluído com feedback humano!")
else:
    print("✅ Transação resolvida automaticamente sem necessidade de interrupção humana.")


## 5. Resultados Experimentais Consolidados do TCC (Capítulo 4)

Resultados **reais** (não mais os valores-guia originais), produzidos pelo pipeline reprodutível em `notebooks/experimentos/` e registrados no MLflow (`experimento_faturas`). Avaliação em **teste-cego por estabelecimento** (334 transações reais C6/Nubank; comércios disjuntos entre treino e teste):

- **Q1 (esparso × denso):** neste acervo de porte reduzido, os modelos **esparsos** (TF-IDF + n-gramas de caractere) foram os mais robustos — TF-IDF+RF com **69,3% de F1-Macro** — à frente da busca vetorial densa (56,2%), do agente híbrido (60,0%) e do ByT5 ajustado (48,3%).
- **Q2 (desempenho global):** o melhor F1-Macro foi **69,3%** e a melhor acurácia **88,3%** (Naive Bayes). **Nenhum modelo atinge 85% de F1-Macro** sob este protocolo; a hipótese central não se confirma na métrica primária (ver Cap. 5 — Limitações).
- A diferença entre validação cruzada 5-fold (RF: 92,7%) e o teste-cego (69,3%) quantifica o efeito de **memorização de comércios já vistos**.


In [ ]:
# Tabela 4.4 do TCC — Comparativo Global (RESULTADOS REAIS)
# Fonte: notebooks/experimentos/ (pipeline reprodutível) -> MLflow experimento_faturas
import pandas as pd
from pathlib import Path

RES = Path("experimentos/resultados")
cons = pd.read_csv(RES / "consolidado_modelos.csv").set_index("modelo")

ORDEM = [
    ("tfidf_naive_bayes", "Baseline 1: TF-IDF + Naive Bayes"),
    ("tfidf_random_forest", "Baseline 2: TF-IDF + Random Forest"),
    ("byt5_finetuning", "SOTA Neural: ByT5 Fine-Tuning (byt5-small)"),
    ("rag_pgvector_minilm", "Busca Vetorial RAG: PGVector (MiniLM)"),
    ("agente_hibrido_langgraph", "Agente Híbrido Completo (LangGraph)"),
]
df_modelos = pd.DataFrame([
    {
        "Modelo / Arquitetura": nome,
        "Acurácia": round(100 * cons.loc[k, "acuracia"], 1),
        "Precisão": round(100 * cons.loc[k, "precisao_macro"], 1),
        "Recall": round(100 * cons.loc[k, "recall_macro"], 1),
        "F1-Macro": round(100 * cons.loc[k, "f1_macro"], 1),
        "Latência (ms)": round(cons.loc[k, "latencia_ms"], 2),
    }
    for k, nome in ORDEM if k in cons.index
])

display(HTML("<h3>Tabela 4.4: Comparativo Global — teste-cego de 334 transações (estabelecimentos disjuntos)</h3>"))
display(df_modelos.style.format({"Acurácia": "{:.1f}%", "Precisão": "{:.1f}%", "Recall": "{:.1f}%",
                                 "F1-Macro": "{:.1f}%", "Latência (ms)": "{:.2f} ms"})
        .highlight_max(subset=["Acurácia", "F1-Macro"], color="#c8e6c9")
        .highlight_min(subset=["Latência (ms)"], color="#d1c4e9"))
print("Nota: nenhum modelo atinge 85% de F1-Macro sob teste-cego por estabelecimento.")
print("CV 5-fold (RF): F1-Macro = %.1f%% -> queda de ~23 p.p. no teste-cego (memorização de comércios)."
      % (100 * cons.loc["tfidf_random_forest", "cv_f1_macro_mean"]))


In [ ]:
# Gráficos comparativos (dados reais)
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
curtos = ["TF-IDF+NB", "TF-IDF+RF", "ByT5", "PGVector", "Agente"][: len(df_modelos)]
x = np.arange(len(df_modelos)); w = 0.35
axes[0].bar(x - w/2, df_modelos["Acurácia"], w, label="Acurácia (%)", color="#3b82f6")
axes[0].bar(x + w/2, df_modelos["F1-Macro"], w, label="F1-Macro (%)", color="#10b981")
axes[0].axhline(y=85.0, color="#ef4444", linestyle="--", label="Meta TCC (85%)")
axes[0].set_xticks(x); axes[0].set_xticklabels(curtos, rotation=15)
axes[0].set_ylabel("%"); axes[0].set_title("Acurácia e F1-Macro (teste-cego real)")
axes[0].set_ylim(0, 100); axes[0].legend(); axes[0].grid(axis="y", linestyle=":", alpha=0.7)

bars = axes[1].bar(curtos, df_modelos["Latência (ms)"],
                   color=["#94a3b8", "#94a3b8", "#f87171", "#34d399", "#60a5fa"][: len(df_modelos)])
axes[1].set_yscale("log"); axes[1].set_ylabel("Latência (ms) — log")
axes[1].set_title("Latência média por transação")
axes[1].set_xticklabels(curtos, rotation=15)
for b in bars:
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height()*1.3, f"{b.get_height():.2f}",
                 ha="center", va="bottom", fontsize=9)
axes[1].grid(axis="y", linestyle=":", alpha=0.7)
plt.tight_layout()
plt.savefig("grafico_comparativo_modelos.png", dpi=200)
plt.show()


In [ ]:
# Tabela 4.5 do TCC — Desempenho por categoria (RESULTADOS REAIS, melhor modelo)
import pandas as pd
from pathlib import Path
RES = Path("experimentos/resultados")

cons = pd.read_csv(RES / "consolidado_modelos.csv").set_index("modelo")
melhor = cons["f1_macro"].idxmax()
df_categorias = pd.read_csv(RES / f"por_categoria__{melhor}.csv")
df_categorias = df_categorias[df_categorias["suporte"] > 0].rename(
    columns={"categoria": "Categoria de Despesa", "precisao": "Precisão",
             "recall": "Recall", "f1": "F1-Score", "suporte": "Volume de Teste"})

plt.figure(figsize=(10, 6))
sns.barplot(data=df_categorias.sort_values("F1-Score"), y="Categoria de Despesa",
            x="F1-Score", hue="Categoria de Despesa", palette="crest", legend=False)
plt.axvline(x=85.0, color="red", linestyle="--", label="Meta 85%")
plt.xlim(0, 100)
plt.title(f"F1-Score por categoria — {melhor} (teste-cego real)")
plt.xlabel("F1-Score (%)"); plt.ylabel(""); plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("grafico_f1_categorias.png", dpi=200)
plt.show()
display(df_categorias)


In [ ]:
# Tabela 4.6 do TCC — Estudo de ablação (RESULTADOS REAIS)
import json, pandas as pd
from pathlib import Path
RES = Path("experimentos/resultados")

abl = json.loads((RES / "ablacao.json").read_text(encoding="utf-8"))
df_ablacao = pd.DataFrame(abl["componentes"]).rename(
    columns={"componente": "Componente", "com": "Com (F1%)", "sem": "Sem (F1%)",
             "delta_f1": "Δ F1 (p.p.)"})
display(df_ablacao)

rot = abl.get("roteamento_agente", {})
print("Roteamento do agente no teste-cego:")
print(f"  ramo vetorial : {100*rot.get('frac_vetorial',0):.0f}% das transações | acurácia {100*rot.get('acuracia_ramo_vetorial',0):.0f}%")
print(f"  ramo LLM local: {100*rot.get('frac_llm',0):.0f}% das transações | acurácia {100*rot.get('acuracia_ramo_llm',0):.0f}%")
print(f"  Human-in-the-Loop: {100*rot.get('frac_hitl',0):.1f}%")
print("\nAchados: normalização léxica ajuda o RAG denso (~+11 p.p.) mas não o TF-IDF;")
print("a busca vetorial é o componente determinante do agente; o LLM local (gemma-4-e4b) sub-desempenha.")


In [ ]:
# Os experimentos reais já ficam registrados no MLflow pelos scripts de notebooks/experimentos/
# (runs cls::tfidf_naive_bayes, cls::tfidf_random_forest, cls::byt5_finetuning,
#  cls::rag_pgvector_minilm, cls::agente_hibrido_langgraph, cls::ablacao_componentes
#  e TCC_Benchmark_Consolidado_Cap4_REAL).
#
# Para reproduzir do zero:
#   cd notebooks/experimentos
#   python 01_preparar_dados.py
#   python 02_baselines.py
#   python 03_byt5.py
#   python 04_rag_pgvector.py --persistir-bd
#   python 05_agente.py --teste            # teste-cego (métricas)
#   python 05_agente.py --gravar --limite 800   # grava predições no Postgres
#   python 06_consolidar.py
#   python 07_ablacao.py
#   python 08_montar_tex.py                # gera os blocos LaTeX do Cap.4
import json
from pathlib import Path
fatos = json.loads((Path("experimentos/resultados") / "fatos_cap4.json").read_text(encoding="utf-8"))
print(json.dumps(fatos["modelos"], indent=2, ensure_ascii=False))
print(f"\nMLflow: {MLFLOW_URI}/#/experiments/1")


## 6. Considerações Finais e Verificação da Hipótese

- **Extração (OCR):** o IBM Docling V2 reduziu o CER de 52,8% (Tesseract) para **5,72%** e elevou o casamento exato de 3,3% para 86,2% sobre 1.398 transações auditadas — resultado **confirmado**.
- **Classificação:** sob teste-cego por estabelecimento, os baselines esparsos clássicos foram os mais robustos (TF-IDF+RF: **69,3% de F1-Macro**); busca vetorial densa 56,2%, agente híbrido 60,0%, ByT5 48,3%. **Nenhum modelo atingiu 85% de F1-Macro.**
- **Veredito da hipótese central:** **não confirmada** na métrica primária (F1-Macro > 85%); parcialmente sustentada em acurácia global (Naive Bayes: 88,3%). Causas: porte reduzido do acervo pessoal rotulado (1.660 transações, 11 categorias), cauda longa de categorias (4–5 exemplos), rotulagem de referência semiautomática e capacidade limitada do LLM local.
- **Ablação:** a busca vetorial é o componente determinante do agente (acurácia condicional 98% no ramo vetorial × 21% no ramo LLM); a normalização léxica é decisiva para embeddings densos (+11 p.p.) mas não para modelos esparsos.
- **MLOps:** todos os experimentos registrados e reproduzíveis no MLflow; Human-in-the-Loop acionado em 4,5% das transações; a busca web (DuckDuckGo) esteve indisponível no ambiente.

> Pipeline reprodutível completo em `notebooks/experimentos/` (`01`–`08`). Blocos LaTeX do Cap. 4 gerados em `notebooks/experimentos/resultados/latex/`.
